# Predicting CANDOR survey responses based on growth rates $\alpha$

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from scipy.stats import chisquare, skewtest, t

# stats in R
import rpy2.robjects as ro
from rpy2.robjects import Formula
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import pandas2ri, default_converter, conversion

import warnings; warnings.filterwarnings('ignore')

In [2]:
vars = [
    ### Affective motive
    # 'best_affect', 'conv_leader', 'conversationalist', 'how_enjoyable', 'i_like_you', 'i_felt_close_to_my_partner', 'i_think_your_status', 'i_would_like_to_become_friends', 'overall_affect',
    # 'responsive_1','responsive_2','responsive_3',
    # 'you_are_good_listener', 'you_are_kind', 'you_are_quickwitted','you_are_warm','you_like_me',
    #test affectives!
    # 'you_are_intelligent', 'you_are_friendly', 'in_common',

    ### Cognitive motives
    # 'anticipated_each_other_sr6', 'developed_joint_perspective_sr2', 'thoughts_became_more_alike_sr5',
    #test cognitives!
    'shared_reality', 'our_thoughts_synced_up_sr1', 'saw_world_in_same_way_sr8',
]

Data paths used in the project

In [3]:
subcorpora_col = 'corpus'

SLOPES_PATH = 'reports/all.parquet'

candor_meta_data_path = '../../updated-data/all-candor-meta_data.csv'

TABLES_FIGS_PATH = 'tables_and_figures/'
if not os.path.exists(TABLES_FIGS_PATH):
    os.mkdir(TABLES_FIGS_PATH)

PARAMETER_ESTIMATES_PATH = 'params/'
if not os.path.exists(PARAMETER_ESTIMATES_PATH):
    os.mkdir(PARAMETER_ESTIMATES_PATH)

if not os.path.exists(os.path.join(TABLES_FIGS_PATH, 'individual_var_stats')):
    os.mkdir(os.path.join(TABLES_FIGS_PATH, 'individual_var_stats'))

## Importing data

In [4]:
results = pd.read_parquet(SLOPES_PATH)
results.head()

,corpus,lang,cid,x_speaker,y_speaker,self,d0,lambda,k
0,CORAAL,eng,CORAAL-CORAAL-se3_ag1_f_01DTA-se3-ag1,CORAAL-CORAAL-se3_ag1_f_01DTA-se3-ag1-PAR0,CORAAL-CORAAL-se3_ag1_f_01DTA-se3-ag1-PAR1,False,0.260724,-0.000483,12891
1,CORAAL,eng,CORAAL-CORAAL-se3_ag1_f_01DTA-se3-ag1,CORAAL-CORAAL-se3_ag1_f_01DTA-se3-ag1-PAR0,CORAAL-CORAAL-se3_ag1_f_01DTA-se3-ag1-PAR2,False,0.132652,0.000000,55
2,CORAAL,eng,CORAAL-CORAAL-se3_ag1_f_01DTA-se3-ag1,CORAAL-CORAAL-se3_ag1_f_01DTA-se3-ag1-PAR1,CORAAL-CORAAL-se3_ag1_f_01DTA-se3-ag1-PAR0,False,0.380026,-0.001738,109001
3,CORAAL,eng,CORAAL-CORAAL-se3_ag1_f_01DTA-se3-ag1,CORAAL-CORAAL-se3_ag1_f_01DTA-se3-ag1-PAR1,CORAAL-CORAAL-se3_ag1_f_01DTA-se3-ag1-PAR2,False,0.391480,-0.001785,1260
4,CABNC,eng,CABNC-CABNC-KCDRE022-KCD,CABNC-CABNC-KCDRE022-KCD-PS0E8,CABNC-CABNC-KCDRE022-KCD-PS6ST,False,2.957569,-0.072520,45


##### Some preliminary statistics

In [5]:
print((results['lambda'].abs() == np.inf).sum(), (results['lambda'].abs() == np.inf).mean())
# results.isna().sum()

0 0.0


### Removing unnecessary data

In [6]:
sel = results['lambda'].isna() | (results['lambda'].abs() == np.inf)
sel.sum(), sel.mean(), (~sel).sum()

(np.int64(0), np.float64(0.0), np.int64(20777))

In [7]:
# is_null = np.array(['null-' in x for x in tqdm(results['x_turn_id'])])
# is_null = results['null']
# is_null.sum()

In [8]:
results = results.loc[
    (~sel)
    # & (~is_null)
    & results[subcorpora_col].isin(['CANDOR'])
]
results.shape

(3306, 9)

In [9]:
results['conversation_id'] = ['-'.join(it.split('-')[3:]) for it in tqdm(results['cid'])]

  0%|          | 0/3306 [00:00<?, ?it/s]

In [10]:
results.head()

,corpus,lang,cid,x_speaker,y_speaker,self,d0,lambda,k,conversation_id
6,CANDOR,eng,CANDOR-CANDOR-20-0235bac2-3469-447b-adae-e3955...,5977e3867412f8000194e1fe,5de33ff0cc246f3633dc01d2,False,0.864507,-0.001154,6650,0235bac2-3469-447b-adae-e39559adfa4a
7,CANDOR,eng,CANDOR-CANDOR-20-0235bac2-3469-447b-adae-e3955...,5de33ff0cc246f3633dc01d2,5977e3867412f8000194e1fe,False,1.642608,0.002188,6750,0235bac2-3469-447b-adae-e39559adfa4a
16,CANDOR,eng,CANDOR-CANDOR-4-19fd3ef7-6ad8-4f2f-a3aa-fd8e2a...,5da8ded81844f20015bf9a62,5ea5ba46461d105ac7fdb4fb,False,2.051832,0.000221,8150,19fd3ef7-6ad8-4f2f-a3aa-fd8e2a0b39cc
17,CANDOR,eng,CANDOR-CANDOR-4-19fd3ef7-6ad8-4f2f-a3aa-fd8e2a...,5ea5ba46461d105ac7fdb4fb,5da8ded81844f20015bf9a62,False,2.931328,-0.000062,8150,19fd3ef7-6ad8-4f2f-a3aa-fd8e2a0b39cc
47,CANDOR,eng,CANDOR-CANDOR-30-ac3a86cc-3a35-4567-acee-752d9...,5e6d6545993427329a35b800,5f5e8c58f7e0497501c4e356,False,0.792172,-0.000761,17350,ac3a86cc-3a35-4567-acee-752d9545ffa3


## Merging with CANDOR metadata

In [11]:
dfm = pd.read_csv(candor_meta_data_path)[['user_id', 'convo_id']+vars]
print(dfm.shape)


(3312, 5)


rescaling data to be in same scale.

In [12]:
for col in tqdm(list(dfm)[2:]):
    dfm[col] = (dfm[col] / dfm[col].max() ) * 10

  0%|          | 0/3 [00:00<?, ?it/s]

In [13]:
dfm.head()

,user_id,convo_id,shared_reality,our_thoughts_synced_up_sr1,saw_world_in_same_way_sr8
0,5efbe4f0c9f37c1a3b70afaf,56f21a75-4e86-4dee-9a15-a4f68e86e1a3,7.678571,7.142857,5.714286
1,5f26378609aca228dcfd19fa,56f21a75-4e86-4dee-9a15-a4f68e86e1a3,8.392857,8.571429,10.000000
2,5d8e361e6e920e0014b459d9,94a1abaa-d89e-4d0d-82ad-1c8de8155a2f,3.392857,1.428571,2.857143
3,5f4d435c030fef9d0136b496,94a1abaa-d89e-4d0d-82ad-1c8de8155a2f,2.857143,2.857143,1.428571
4,5ca6bbf13b5fcf00100996e9,4e3233b7-5d89-47e9-a795-25452f99a1c9,6.428571,5.714286,5.714286


#### Testing for normalcy and uniformity

In [ ]:
normalcy_and_other_stats = []

In [ ]:
for var in vars:
    sel = dfm[var].isna()
    maxima = dfm[var].loc[~sel].max()
    minima = dfm[var].loc[~sel].min()
    medi = dfm[var].loc[~sel].median()
    mu = dfm[var].loc[~sel].mean()
    mode = dfm[var].loc[~sel].mode()
    skew_stat, p_skew = skewtest(dfm[var].loc[~sel].values)

    dv = dfm[var].loc[~sel].value_counts()
    chi_stat, p_chi = chisquare(f_obs=dv.values, f_exp=np.array([1/len(dv)]*len(dv))*dv.sum())

    normalcy_and_other_stats += [
        {
            'variable': var,
            'range': '[{}, {}]'.format(minima, maxima),
            'median': medi.item(),
            'mean': mu.item(),
            'mode': mode.values,
            'skew_stat': skew_stat, 'skew_p': p_skew,
            'chi_stat': chi_stat, 'chi_p': p_chi
        }
    ]

normalcy_and_other_stats = pd.DataFrame(normalcy_and_other_stats)

In [ ]:
normalcy_and_other_stats.head(n=40)

In [ ]:
normalcy_and_other_stats.to_csv('candor-normalcy-stats.csv', index=False, encoding='utf-8')

#### Merging docs

In [14]:
results = pd.merge(
    left=results, left_on=['x_speaker', 'conversation_id'],
    right=dfm, right_on=['user_id', 'convo_id'],
    # left=results, left_on=['speaker'],
    # right=dfm, right_on=['user_id'],
    how='left'
)

In [15]:
results = results.drop(columns=['user_id', 'convo_id'])
# results = results.drop(columns=['user_id'])
# results.columns
results.isna().sum()

corpus                         0
lang                           0
cid                            0
x_speaker                      0
y_speaker                      0
self                           0
d0                             0
lambda                         0
k                              0
conversation_id                0
shared_reality                56
our_thoughts_synced_up_sr1    56
saw_world_in_same_way_sr8     56
dtype: int64

## Predicting individual survey variables from decay rates $\alpha$

### Converting categorical cols to numbered indexes

In [16]:
## convert speaker to integers
col = 'x_speaker'
vs = results[col].unique()
# sid = {sp:i+1 for i,sp in enumerate(np.random.choice(vs,size=(len(vs),),replace=False))}
sid = {sp:i+1 for i,sp in enumerate(vs)}
results[col+'_'] = [sid[sp] for sp in tqdm(results[col])]

  0%|          | 0/3306 [00:00<?, ?it/s]

In [17]:
## convert turn_ids to integers
col = 'conversation_id'
vs = results[col].astype(str).unique()
# sid = {sp:i+1 for i,sp in enumerate(np.random.choice(vs,size=(len(vs),),replace=False))}
sid = {sp:i+1 for i,sp in enumerate(vs)}
results[col+'_'] = [sid[sp] for sp in tqdm(results[col].astype(str))]

  0%|          | 0/3306 [00:00<?, ?it/s]

### Running JAGS model in R

In [18]:
# model_ = str(model_script)

for var in tqdm(vars):
    sel_2 = ~results[var].isna()

    with localconverter(default_converter + pandas2ri.converter):
        df_ = conversion.py2rpy(results[['lambda', 'd0', 'conversation_id_', 'x_speaker_', var]].loc[sel_2])

    ro.globalenv['df'] = df_

    ro.r("""
    library(R2jags)
    # print(colnames(df))
    survey_var <- df${}
    d0 <- df$d0
    slope <- df$lambda
    speaker <- df$x_speaker_
    conversation <- df$conversation_id_
    KSPEAKER <- max(speaker)
    CCONVERSATION <- max(conversation)
    # print(KSPEAKER)
    # print(CCONVERSATION)
    ROWS <- length(slope)

    b0_mu <- mean(survey_var)
    lv <- var(survey_var)

    data <- list(
        'lv',
        'survey_var',
        'slope',
        'd0',
        # 'speaker',
        # 'conversation',
        'b0_mu',
        # 'CCONVERSATION',
        # 'KSPEAKER',
        'ROWS'
    )

    myinits <- list(
        list(
            eta=.5,
            # b0=0.1,
            phi=1.0,
            beta=1.0
        )
    )

    parameters <- c(
        'beta',
        'phi',
        'eta',
        'b0'
    )

    samples<- jags(data, inits=myinits, parameters, model.file='model.txt', n.chains=1, n.iter=5000, n.burnin=100, DIC=T, progress.bar="text")

    beta <- samples$BUGSoutput$sims.list$beta
    b0 <- samples$BUGSoutput$sims.list$b0
    eta <- samples$BUGSoutput$sims.list$eta
    phi <- samples$BUGSoutput$sims.list$phi
    save(
        beta,
        b0,
        phi,
        eta,
        file='{}/{}.RData'
    )
    """.format(var, PARAMETER_ESTIMATES_PATH, var))

  0%|          | 0/3 [00:00<?, ?it/s]

R callback write-console: Loading required package: rjags
  
R callback write-console: Loading required package: coda
  
R callback write-console: Linked to JAGS 4.3.2
  
R callback write-console: Loaded modules: basemod,bugs
  
R callback write-console: 
Attaching package: ‘R2jags’

  
R callback write-console: The following object is masked from ‘package:coda’:

    traceplot

  
R callback write-console: module glm loaded
  


Compiling model graph
   Resolving undeclared variables
   Allocating nodes
Graph information:
   Observed stochastic nodes: 3250
   Unobserved stochastic nodes: 4
   Total graph size: 22766

Initializing model

  |**************************************************| 100%
  |**************************************************| 100%
Compiling model graph
   Resolving undeclared variables
   Allocating nodes
Graph information:
   Observed stochastic nodes: 3250
   Unobserved stochastic nodes: 4
   Total graph size: 22766

Initializing model

  |**************************************************| 100%
  |**************************************************| 100%
Compiling model graph
   Resolving undeclared variables
   Allocating nodes
Graph information:
   Observed stochastic nodes: 3250
   Unobserved stochastic nodes: 4
   Total graph size: 22766

Initializing model

  |**************************************************| 100%
  |**************************************************| 100%
